In [12]:
import numpy as np
import pandas as pd

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [5]:
df_acc_fet = pd.read_csv('account_features(1).csv')

In [10]:
df_acc_fet.head()

,customer_id,device_degree,ip_velocity_10m_max,contagion,connected_customers,unique_devices,unique_ips,unique_addresses,unique_payments,transaction_count,return_rate,observed_dispute_count
0,cust_01571,3,1,0.021739,44,1,5,1,5,5,0.2,0
1,cust_08153,1,1,0.032787,59,1,6,1,6,6,0.0,0
2,cust_02630,3,1,0.027778,34,2,4,1,4,4,0.0,0
3,cust_11379,2,1,0.031915,92,1,10,1,10,10,0.1,1
4,cust_08043,1,1,0.066667,28,1,3,1,3,3,0.0,0


In [11]:
df_acc_fet.info()

<class 'pandas.DataFrame'>
RangeIndex: 11849 entries, 0 to 11848
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             11849 non-null  str    
 1   device_degree           11849 non-null  int64  
 2   ip_velocity_10m_max     11849 non-null  int64  
 3   contagion               11849 non-null  float64
 4   connected_customers     11849 non-null  int64  
 5   unique_devices          11849 non-null  int64  
 6   unique_ips              11849 non-null  int64  
 7   unique_addresses        11849 non-null  int64  
 8   unique_payments         11849 non-null  int64  
 9   transaction_count       11849 non-null  int64  
 10  return_rate             11849 non-null  float64
 11  observed_dispute_count  11849 non-null  int64  
dtypes: float64(2), int64(9), str(1)
memory usage: 1.1 MB


In [12]:
df_acc_fet.isnull().sum()

customer_id               0
device_degree             0
ip_velocity_10m_max       0
contagion                 0
connected_customers       0
unique_devices            0
unique_ips                0
unique_addresses          0
unique_payments           0
transaction_count         0
return_rate               0
observed_dispute_count    0
dtype: int64

In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, average_precision_score, confusion_matrix
import shap
import joblib

In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, average_precision_score, confusion_matrix
import shap
import joblib


In [6]:
features_df = pd.read_csv("data1/account_features.csv")
transactions_df = pd.read_csv("data1/transactions.csv") 

In [7]:
features_df.head()

,customer_id,device_degree,ip_velocity_10m_max,contagion,connected_customers,unique_devices,unique_ips,unique_addresses,unique_payments,transaction_count,return_rate,observed_dispute_count
0,cust_01571,3,1,0.021739,44,1,5,1,5,5,0.2,0
1,cust_08153,1,2,0.032787,59,1,6,1,6,6,0.0,0
2,cust_02630,3,3,0.027778,34,2,4,1,4,4,0.0,0
3,cust_11379,2,2,0.031915,92,1,10,1,10,10,0.1,1
4,cust_08043,1,4,0.066667,28,1,3,1,3,3,0.0,0


In [9]:
features_df.sample(10)

,customer_id,device_degree,ip_velocity_10m_max,contagion,connected_customers,unique_devices,unique_ips,unique_addresses,unique_payments,transaction_count,return_rate,observed_dispute_count
636,cust_09821,2,1,0.028571,33,2,5,1,5,5,0.000000,0
998,cust_08236,1,1,0.062500,30,1,4,1,4,4,0.000000,0
11436,cust_03248,1,1,0.058824,15,1,2,1,2,2,0.000000,0
9884,cust_02532,1,1,0.039216,49,1,6,1,6,6,0.166667,0
4536,cust_02484,1,3,0.018182,53,1,7,1,7,7,0.000000,0
10585,cust_04353,1,1,0.025000,38,1,5,1,5,5,0.200000,0
937,cust_05160,1,2,0.117647,32,1,4,1,4,4,0.000000,0
4355,cust_05052,1,1,0.103448,27,1,4,1,4,4,0.000000,0
9916,cust_08985,2,1,0.054054,35,1,4,1,4,4,0.000000,0
3020,cust_03775,1,2,0.093750,30,1,4,1,4,4,0.000000,0


In [14]:
transactions_df.head()

,transaction_id,customer_id,timestamp,amount,device_id,ip_id,address_id,payment_id,product_id,status,returned,dispute,is_fraud_ring,ring_id
0,txn_0021361,cust_01571,2026-01-01T00:03:20+00:00,1750.67,dev_01571,ip_11087,addr_10997,pay_09289,sku_1449,captured,0,0,0,NaN
1,txn_0005431,cust_08153,2026-01-01T00:03:33+00:00,838.66,dev_08153,ip_00023,addr_02071,pay_01877,sku_1661,captured,0,0,0,NaN
2,txn_0024368,cust_02630,2026-01-01T00:07:47+00:00,808.60,dev_07520,ip_13813,addr_07410,pay_06762,sku_0295,captured,0,0,0,NaN
3,txn_0020942,cust_11379,2026-01-01T00:10:14+00:00,2869.14,dev_00879,ip_05130,addr_02653,pay_01598,sku_1821,captured,1,0,0,NaN
4,txn_0002922,cust_08043,2026-01-01T00:14:05+00:00,2631.16,dev_08043,ip_07539,addr_01301,pay_07580,sku_2359,captured,0,0,0,NaN


In [16]:
data = pd.merge(features_df, labels, on="customer_id", how="inner")

In [15]:
labels = transactions_df.groupby("customer_id")["is_fraud_ring"].max().reset_index()

In [78]:
feature_cols = [
    "device_degree",
    "ip_velocity_10m_max",
    "contagion",
    "connected_customers",
    "unique_devices",
    "unique_ips",
    "unique_addresses",
    "unique_payments",
    "transaction_count",
    "return_rate",
    "observed_dispute_count"
]


In [80]:
x = data[feature_cols].copy()

In [79]:
x = data["is_fraud_ring"].copy()

In [81]:
x

,device_degree,ip_velocity_10m_max,contagion,connected_customers,unique_devices,unique_ips,unique_addresses,unique_payments,transaction_count,return_rate,observed_dispute_count
0,3,1,0.021739,44,1,5,1,5,5,0.2,0
1,1,2,0.032787,59,1,6,1,6,6,0.0,0
2,3,3,0.027778,34,2,4,1,4,4,0.0,0
3,2,2,0.031915,92,1,10,1,10,10,0.1,1
4,1,4,0.066667,28,1,3,1,3,3,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...
11844,2,1,0.090909,9,1,1,1,1,1,0.0,0
11845,4,1,0.100000,8,1,1,1,1,1,0.0,0
11846,1,1,0.100000,8,1,1,1,1,1,0.0,0
11847,1,1,0.125000,6,1,1,1,1,1,0.0,0


In [68]:
y = data.iloc[:,-1]

In [82]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [83]:
model = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4,
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train), 
        eval_metric="logloss",
        random_state=42
    )

In [84]:
model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [85]:
x.dtypes

device_degree               int64
ip_velocity_10m_max         int64
contagion                 float64
connected_customers         int64
unique_devices              int64
unique_ips                  int64
unique_addresses            int64
unique_payments             int64
transaction_count           int64
return_rate               float64
observed_dispute_count      int64
dtype: object

In [86]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [87]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
pr_auc = average_precision_score(y_test, y_prob)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("Precision:", precision)
print("Recall:", recall)
print("PR-AUC:", pr_auc)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

Precision: 1.0
Recall: 1.0
PR-AUC: 1.0
TN: 3540
FP: 0
FN: 0
TP: 15


In [75]:
y.sample(10)

1923    0
6422    0
7738    0
5719    0
8402    0
7493    0
3495    0
9945    0
4979    0
6604    0
Name: is_fraud_ring, dtype: int64

In [76]:
y_prob

array([0.00011846, 0.00011846, 0.00011846, ..., 0.00011846, 0.00011846,
       0.00011846], shape=(3555,), dtype=float32)

In [88]:
print(x.columns.tolist())

['device_degree', 'ip_velocity_10m_max', 'contagion', 'connected_customers', 'unique_devices', 'unique_ips', 'unique_addresses', 'unique_payments', 'transaction_count', 'return_rate', 'observed_dispute_count']


In [89]:
feature_cols = [
    "device_degree",
    "ip_velocity_10m_max",
    "contagion",
    "connected_customers",
    "unique_devices",
    "unique_ips",
    "unique_addresses",
    "unique_payments",
    "transaction_count",
    "return_rate",
    "observed_dispute_count"
]

X = data[feature_cols].copy()
y = data["is_fraud_ring"].copy()

print("X columns:")
print(X.columns.tolist())

print("\nX shape:", X.shape)
print("\ny distribution:")
print(y.value_counts())

X columns:
['device_degree', 'ip_velocity_10m_max', 'contagion', 'connected_customers', 'unique_devices', 'unique_ips', 'unique_addresses', 'unique_payments', 'transaction_count', 'return_rate', 'observed_dispute_count']

X shape: (11849, 11)

y distribution:
is_fraud_ring
0    11799
1       50
Name: count, dtype: int64


In [90]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\ny_train:")
print(y_train.value_counts())

print("\ny_test:")
print(y_test.value_counts())

X_train: (9479, 11)
X_test: (2370, 11)

y_train:
is_fraud_ring
0    9439
1      40
Name: count, dtype: int64

y_test:
is_fraud_ring
0    2360
1      10
Name: count, dtype: int64


In [91]:
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    scale_pos_weight=(len(y_train) - y_train.sum()) / y_train.sum(),
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [92]:
importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance)

ip_velocity_10m_max       1.0
device_degree             0.0
contagion                 0.0
connected_customers       0.0
unique_devices            0.0
unique_ips                0.0
unique_addresses          0.0
unique_payments           0.0
transaction_count         0.0
return_rate               0.0
observed_dispute_count    0.0
dtype: float32


In [93]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
pr_auc = average_precision_score(y_test, y_prob)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_pred
).ravel()

print("Precision:", precision)
print("Recall:", recall)
print("PR-AUC:", pr_auc)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

Precision: 1.0
Recall: 1.0
PR-AUC: 1.0
TN: 2360
FP: 0
FN: 0
TP: 10
